In [1]:
import faultier
import serial
ft = faultier.Faultier()
ser = serial.Serial(ft.get_serial_path(), baudrate=115200)
ser.reset_input_buffer()
ser.timeout = 0.3


ft.configure_glitcher(
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000
)

ft.power_cycle()
print(ser.read(100))

b'\x00RSTUUUU'


In [3]:
import struct
from tqdm.notebook import trange
ft.configure_glitcher(
    trigger_source = faultier.TRIGGER_IN_EXT0,
    trigger_type = faultier.TRIGGER_PULSE_POSITIVE,
    glitch_output= faultier.OUT_CROWBAR
)

for d in trange(0, 20000):
    for p in range(0, 9):
        ser.reset_input_buffer()
        ft.glitch(delay=d, pulse=p)
        data = ser.read(8) # 0 byte + RST + 4 byte number
        if(len(data)) != 8:
            continue
        if(data[1:4] != b"RST"):
            pass
            continue
        if(data[4:8] == b"RST") or (data[4:8] == b"\x00RS"):
            # print("Reset {d} {p}")
            continue
        if b"\xaa\xaa\xaa\xaa" in data:
            print(f"{d} {p} {data.hex()}")
        if(data[4:9] != b"\x55\x55\x55\x55"):
            value = struct.unpack("<I", data[4:])[0]
            if value == 0x54535200:
                # print(f"Reset {d} {p}")
                continue
            print(f"{d} {p} {hex(value)}")

  0%|          | 0/20000 [00:00<?, ?it/s]

13 7 0x55545352
45 6 0x55545352
76 7 0x55545352
99 6 0x1234
103 7 0x1234
123 6 0x1234
127 6 0x55545352
131 6 0x55545352
133 7 0xaaaaab54
139 5 0xaaaaaaff
139 6 0xaaaaaaff
140 5 0xaaaaaaff
140 6 0xaaaaaaff
142 5 0xaaaaaaff
142 6 0xaaaaaaff
143 5 0xaaaaaaff
143 6 0xaaaaaaff
149 6 00525354aaaaaaaa
149 6 0xaaaaaaaa
154 7 00525354aaaaaaaa
154 7 0xaaaaaaaa
159 6 0x555555aa
162 8 00525354aaaaaaaa
162 8 0xaaaaaaaa
163 5 0x575555aa
163 6 0x410059
164 5 0x554551aa
197 6 0x55545352
212 6 0x0
226 8 0x55545352
248 6 0x55545352
259 7 0x55545352
264 6 0x1234


KeyboardInterrupt: 

In [79]:
import struct
from tqdm.notebook import trange
ft.configure_glitcher(
    trigger_source = faultier.TRIGGER_IN_EXT0,
    trigger_type = faultier.TRIGGER_PULSE_POSITIVE,
    glitch_output= faultier.OUT_CROWBAR
)

ser.reset_input_buffer()
ft.glitch(delay=156, pulse=6)
data = ser.read(8) # 0 byte + RST + 4 byte number
if(data[1:4] != b"RST"):
    pass
if(data[4:8] == b"RST") or (data[4:8] == b"\x00RS"):
    print("Reset {d} {p}")
if(data[4:9] != b"\x55\x55\x55\x55"):
    value = struct.unpack("<I", data[4:])[0]
    if value == 0x54535200:
        print(f"Reset {d} {p}")
    print(f"{d} {p} {hex(value)}")